# 02 — Features: Design Matrix and Group-Aligned X

**Project H18 — Compensation Equity Analyzer.** We build the OLS design matrix exactly as the model uses it (`comp_equity.features.build_design_matrix`) and align the M and F group matrices for the Blinder-Oaxaca decomposition.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
sys.path.insert(0, '../src')
from comp_equity.features import build_design_matrix, align_columns, NUMERIC, CATEGORICAL
df = pd.read_parquet('../data/processed/org_frame.parquet')
len(df)

## 1. Build X, y

In [ ]:
X, y, names = build_design_matrix(df)
print(f'design matrix: {X.shape}')
print(f'feature names (first 8): {names[:8]}')
print(f'y = log(monthly_comp_aed)   mean={y.mean():.3f}   std={y.std():.3f}')

## 2. Histogram of log-comp

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(y, bins=40, color='#1f77b4', ax=ax)
ax.set_title('Distribution of log monthly comp (the OLS target)')
plt.tight_layout(); plt.show()

## 3. Per-group design matrix (M vs F) — alignment

In [ ]:
a_mask = (df['gender'] == 'M').values
b_mask = (df['gender'] == 'F').values
XA = X.loc[a_mask]; XB = X.loc[b_mask]
XA, XB = align_columns(XA, XB)
print(f'M block: {XA.shape}    F block: {XB.shape}')
print(f'columns identical: {list(XA.columns) == list(XB.columns)}')

## 4. Mean feature vector per group

In [ ]:
mu_a = XA.mean(); mu_b = XB.mean()
diffs = pd.DataFrame({'mu_M': mu_a, 'mu_F': mu_b, 'diff': mu_a - mu_b})
diffs = diffs.iloc[1:]  # drop intercept
print(diffs.round(3).head(15))

## 5. Top features by mean-difference (composition channels)

In [ ]:
top = diffs.reindex(diffs['diff'].abs().sort_values(ascending=False).index).head(12)
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#fb6a4a' if v < 0 else '#1f77b4' for v in top['diff']]
ax.barh(top.index, top['diff'], color=colors)
ax.axvline(0, color='black', lw=0.6)
ax.set_title('Top mean-feature differences (M − F) — composition channels')
plt.tight_layout(); plt.show()

## 6. Numeric feature distributions per gender

In [ ]:
fig, axes = plt.subplots(1, len(NUMERIC), figsize=(16, 3.2))
for ax, f in zip(axes, NUMERIC):
    sns.boxplot(data=df, x='gender', y=f, palette=['#1f77b4', '#fb6a4a'], ax=ax)
    ax.set_title(f); ax.set_xlabel('')
plt.tight_layout(); plt.show()

## 7. Multicollinearity check — VIF on a small subset

In [ ]:
subset = NUMERIC
Xs = df[subset].astype(float)
vifs = []
for col in subset:
    Xc = Xs.drop(columns=[col]).values
    yc = Xs[col].values
    beta = np.linalg.lstsq(np.column_stack([Xc, np.ones(len(Xc))]), yc, rcond=None)[0]
    yhat = np.column_stack([Xc, np.ones(len(Xc))]) @ beta
    r2 = 1 - ((yc - yhat) ** 2).sum() / ((yc - yc.mean()) ** 2).sum()
    vifs.append(dict(feature=col, VIF=1.0 / max(1 - r2, 1e-6)))
print(pd.DataFrame(vifs).round(2))

## 8. Save the engineered design matrix

In [ ]:
out = Path('../data/processed/design_matrix.parquet')
X_save = X.copy()
X_save['gender'] = df['gender'].values
X_save['log_comp'] = y
X_save.to_parquet(out, index=False)
print(f'wrote {X_save.shape} -> {out}')

## 9. Implications
- The biggest mean-feature differences between M and F are at the role / level boundary — that is the *composition* channel and will land in the E (endowment) component.
- Pure numeric collinearity is moderate (tenure ↔ level), within tolerable range for OLS with the small ridge term in `fit_ols`.